# Asistente RAG sobre los Planes Nacionales de Desarrollo (2006-2026)
**Curso:** Procesamiento de Lenguaje Natural·

**Equipo:**
*A. Rubiano Carlos*,
*Barrios Alexandra*

**Versión:** 5

**Qué hace.** Dada una pregunta, busca los párrafos más parecidos en los PND de 5 gobiernos y responde **citando gobierno, documento y página**.
Si no encuentra evidencia, lo dice.

**Problema de la v4.** Las categorías salían de un diccionario de palabras clave y un modelo aprendía a predecirlas, así que una exactitud de 88,7 % solo
mostraba que el modelo repetía la regla. En esta versión **no se reporta ninguna exactitud contra esas etiquetas**. El sistema se evalúa con
**preguntas escritas por personas, con el documento y las páginas donde está la respuesta** (`data/gold/preguntas_rag.csv`), partidas en `dev` (para decidir) y
`test` (reservado, se mira una sola vez).

| Sección | Qué se hace | Clase |
|---|---|---|
| 1-3 | Datos: descarga, texto, párrafos (de la v4) | — |
| 4-5 | Embeddings + índice FAISS; búsqueda | S14_1 |
| 6 | **Evaluación independiente** (TF-IDF vs. denso, umbral, test) | S13_2 |
| 7 | RAG: respuesta extractiva con citas y, opcional, con LLM | S13_2 · S14_1 |
| 8-9 | Rúbrica manual y chat | S14_2 |

⚠️ En Colab, `/content` se borra al cerrar la sesión. Los PDF y el índice van a Drive; `data/gold/` hay que subirlo a Git o guardarlo (última sección).

## 1 · Configuración (Colab)
Activa la GPU: *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.

In [1]:
import os, sys

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/asynchronousUC/An-lisis-documentos-DNP"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if EN_COLAB:
    # Limpia cualquier clonación duplicada previa y vuelve a /content
    %cd /content
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

    if os.path.exists("requirements.txt"):
        !pip install -q -r requirements.txt
    else:
        print("⚠️ Aún no existe requirements.txt en la raíz del repositorio.")

    os.makedirs("/content/dnp_rag_data", exist_ok=True)
    os.environ["DNP_DATA_DIR"] = "/content/dnp_rag_data"

sys.path.append(os.path.join(REPO_DIR, "src") if EN_COLAB else "src")

/content
Cloning into '/content/An-lisis-documentos-DNP'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 39 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 1.83 MiB | 12.91 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/An-lisis-documentos-DNP
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


### **En caso tal:** Se deba limpiar y clonar la carpeta en GitHub desde cero

In [ ]:
%cd /content
!rm -rf /content/An-lisis-documentos-DNP
!git clone https://github.com/asynchronousUC/An-lisis-documentos-DNP /content/An-lisis-documentos-DNP
%cd /content/An-lisis-documentos-DNP

/content
Cloning into '/content/An-lisis-documentos-DNP'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 39 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 1.83 MiB | 14.86 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/An-lisis-documentos-DNP


### **Actualización:** Descargar los cambios nuevos y verificar dónde quedó el archivo

In [ ]:
%cd /content/An-lisis-documentos-DNP
!git fetch --all
!git pull
!ls -la

/content/An-lisis-documentos-DNP
Already up to date.
total 2252
drwxr-xr-x 3 root root    4096 Nov 24 11:40 .
drwxr-xr-x 1 root root    4096 Nov 24 11:40 ..
-rw-r--r-- 1 root root 2274327 Nov 24 11:40 DNP_NLP_v4.ipynb
drwxr-xr-x 8 root root    4096 Nov 24 11:40 .git
-rw-r--r-- 1 root root     303 Nov 24 11:40 .gitignore
-rw-r--r-- 1 root root    7182 Nov 24 11:40 README.md
-rw-r--r-- 1 root root     761 Nov 24 11:40 requirements.txt


In [2]:
import os
import sys
import zipfile

# 1. Descomprimir el archivo zip que contiene el módulo dnp_rag si existe
zip_path = "/content/dnp-rag.zip"
dest_path = "/content/An-lisis-documentos-DNP"

if os.path.exists(zip_path):
    print(f"Descomprimiendo {zip_path} en {dest_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dest_path)

# 2. Buscar dinámicamente la ruta que contiene el paquete 'dnp_rag'
# El zip extraído crea una estructura que puede contener la carpeta del paquete.
posibles_rutas = [
    dest_path,
    os.path.join(dest_path, "dnp-rag"),
    os.path.join(dest_path, "dnp-rag", "src")
]

for ruta in posibles_rutas:
    if os.path.exists(ruta) and ruta not in sys.path:
        sys.path.append(ruta)

import pandas as pd
from IPython.display import display

# 3. Intentar la importación del paquete dnp_rag
from dnp_rag import (config, descarga, diccionario, evaluacion, extraccion, indice, rag,
                     recuperacion, segmentacion)

print("¡Importación exitosa!")
print("Datos en:", config.DATA_DIR)
print("Gold en :", config.GOLD_DIR)

Descomprimiendo /content/dnp-rag.zip en /content/An-lisis-documentos-DNP...
¡Importación exitosa!
Datos en: /content/dnp_rag_data
Gold en : /content/An-lisis-documentos-DNP/dnp-rag/data/gold


## 2 · Datos: PDF → texto → párrafos
**Versión 4** ahora en módulos. Cada etapa se salta si su resultado ya existe.
*Prueba rápida con un solo gobierno:* `PERIODOS_PRUEBA = ["2022-2026"]` (después, vuelve a `None` y ejecuta de nuevo para tener los cinco).

### **❌ Bloqueado desde Colab:** El servidor del DNP está bloqueando la IP de Google Colab (geobloqueo)

In [ ]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    r = requests.get("https://www.dnp.gov.co/plan-nacional-desarrollo/Paginas/default.aspx", headers=headers, timeout=10)
    print("✅ Conexión exitosa. Código:", r.status_code)
except Exception as e:
    print("❌ Bloqueado desde Colab:", type(e).__name__, e)

✅ Conexión exitosa. Código: 200


In [3]:
import shutil
import zipfile
from pathlib import Path
from requests.exceptions import RequestException

PERIODOS_PRUEBA = None
ZIP_RESPALDO = Path("/content/pnd_seleccion.zip")  # Ajusta la ruta si lo tienes en Google Drive

def cargar_desde_zip(zip_path: Path, destino_pdfs: Path):
    """Extrae pnd_seleccion.zip y normaliza las subcarpetas a 'AAAA-AAAA' dentro de config.PDF_DIR."""
    if not zip_path.exists():
        raise FileNotFoundError(
            f"❌ Colab está bloqueado por el DNP y no se encontró el respaldo en: {zip_path}\n"
            "Sube 'pnd_seleccion.zip' a /content (o ajusta ZIP_RESPALDO) y vuelve a ejecutar."
        )

    temp_dir = Path("/content/_temp_pnd_zip")
    if temp_dir.exists():
        shutil.rmtree(temp_dir)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(temp_dir)

    destino_pdfs.mkdir(parents=True, exist_ok=True)
    copiados = 0
    for pdf in temp_dir.rglob("*.pdf"):
        # Toma los primeros 9 caracteres de la carpeta padre (ej. '2006-2010 - Álvaro...' -> '2006-2010')
        carpeta_padre = pdf.parent.name
        periodo = carpeta_padre[:9] if len(carpeta_padre) >= 9 and carpeta_padre[4] == "-" else carpeta_padre

        carpeta_destino = destino_pdfs / periodo
        carpeta_destino.mkdir(parents=True, exist_ok=True)
        shutil.copy2(pdf, carpeta_destino / pdf.name)
        copiados += 1

    shutil.rmtree(temp_dir)
    print(f"✅ Respaldo cargado: {copiados} PDFs organizados en {destino_pdfs}")

# 1. Descarga con fallback automático a pnd_seleccion.zip si hay bloqueo
if not any(config.PDF_DIR.rglob("*.pdf")):
    try:
        descarga.descargar_todo(PERIODOS_PRUEBA)
    except (RequestException, TimeoutError, ConnectionError) as e:
        print(f"⚠️ Bloqueado desde Colab ({type(e).__name__}). Usando respaldo local...")
        cargar_desde_zip(ZIP_RESPALDO, config.PDF_DIR)

# 2. Extracción y segmentación (sin cambios)
if not (config.CORPUS_DIR / "corpus_paginas.jsonl").exists():
    display(extraccion.extraer_todo())
if not (config.CORPUS_DIR / "parrafos.csv").exists():
    _, resumen = segmentacion.segmentar_todo()
    display(resumen)

Gobiernos a revisar: ['2022-2026 - Gustavo Petro', '2018-2022 - Iván Duque Márquez', '2014-2018 - Juan Manuel Santos', '2010-2014 - Juan Manuel Santos', '2006-2010 - Álvaro Uribe Vélez']
2022-2026 - Gustavo Petro: 22 PDF(s)
2018-2022 - Iván Duque Márquez: 17 PDF(s)
2014-2018 - Juan Manuel Santos: 4 PDF(s)
2010-2014 - Juan Manuel Santos: 25 PDF(s)
2006-2010 - Álvaro Uribe Vélez: 40 PDF(s)
[1/9] OK  plan-nacional-de-desarrollo-2022-2026-colombia-potencia-mundial-de-la-vida.pdf
[2/9] OK  tomo-1-dnp.pdf
[3/9] OK  tomo-2-pnd.pdf
[4/9] OK  PND 2014-2018 Tomo 1 internet.pdf
[5/9] OK  PND 2014-2018 Tomo 2 internet.pdf
[6/9] OK  PND2010-2014 Tomo I CD.pdf
[7/9] OK  PND2010-2014 Tomo II CD.pdf
[8/9] OK  PND_Tomo_1.pdf
[9/9] OK  PND_Tomo_2.pdf

Descargados: 9 | Fallidos: 0 | Carpeta: /content/dnp_rag_data/pdfs
✔ 2006-2010 | PND_Tomo_1.pdf | 440 págs | 21 sin texto
✔ 2006-2010 | PND_Tomo_2.pdf | 300 págs | 16 sin texto
✔ 2010-2014 | PND2010-2014 Tomo I CD.pdf | 541 págs | 12 sin texto
✔ 2010-2014 

,periodo,documento,paginas,paginas_sin_texto,pct_sin_texto
0,2006-2010,PND_Tomo_1,440,21,4.8
1,2006-2010,PND_Tomo_2,300,16,5.3
2,2010-2014,PND2010-2014 Tomo I CD,541,12,2.2
3,2010-2014,PND2010-2014 Tomo II CD,481,14,2.9
4,2014-2018,PND 2014-2018 Tomo 1 internet,550,26,4.7
5,2014-2018,PND 2014-2018 Tomo 2 internet,680,20,2.9
6,2018-2022,tomo-1-dnp,910,29,3.2
7,2018-2022,tomo-2-pnd,1050,40,3.8
8,2022-2026,plan-nacional-de-desarrollo-2022-2026-colombia...,848,19,2.2


✔ 2006-2010 | PND_Tomo_1: 2619 segmentos, 12 páginas excluidas
✔ 2006-2010 | PND_Tomo_2: 1602 segmentos, 27 páginas excluidas
✔ 2010-2014 | PND2010-2014 Tomo I CD: 2483 segmentos, 13 páginas excluidas
✔ 2010-2014 | PND2010-2014 Tomo II CD: 2365 segmentos, 29 páginas excluidas
✔ 2014-2018 | PND 2014-2018 Tomo 1 internet: 2653 segmentos, 6 páginas excluidas
✔ 2014-2018 | PND 2014-2018 Tomo 2 internet: 2922 segmentos, 22 páginas excluidas
✔ 2018-2022 | tomo-1-dnp: 4807 segmentos, 8 páginas excluidas
✔ 2018-2022 | tomo-2-pnd: 5158 segmentos, 21 páginas excluidas
✔ 2022-2026 | plan-nacional-de-desarrollo-2022-2026-colombia-potencia-mundial-de-la-vida: 3424 segmentos, 4 páginas excluidas


,periodo,documento,paginas,paginas_excluidas,segmentos
0,2006-2010,PND_Tomo_1,440,12,2619
1,2006-2010,PND_Tomo_2,300,27,1602
2,2010-2014,PND2010-2014 Tomo I CD,541,13,2483
3,2010-2014,PND2010-2014 Tomo II CD,481,29,2365
4,2014-2018,PND 2014-2018 Tomo 1 internet,550,6,2653
5,2014-2018,PND 2014-2018 Tomo 2 internet,680,22,2922
6,2018-2022,tomo-1-dnp,910,8,4807
7,2018-2022,tomo-2-pnd,1050,21,5158
8,2022-2026,plan-nacional-de-desarrollo-2022-2026-colombia...,848,4,3424


## 3 · Vistazo al corpus
Cada fila es un párrafo con su gobierno, documento y **página física del PDF** (la que muestra el visor, no el número impreso al pie): eso es la cita.
La columna `tema` viene del diccionario de la v4 y **solo sirve para explorar**: es una regla, no verdad de referencia.

In [4]:
df = diccionario.etiquetar(pd.read_csv(config.CORPUS_DIR / "parrafos.csv"))
print(f"{len(df):,} párrafos de {df['periodo'].nunique()} gobiernos")
display(df.groupby(["periodo", "presidente"]).agg(parrafos=("id", "size"), paginas=("pagina", "nunique")))
display(diccionario.resumen_temas(df).head(8))
display(df[["periodo", "documento", "pagina", "texto"]].sample(3, random_state=1))

28,033 párrafos de 5 gobiernos


,,parrafos,paginas
periodo,presidente,,
2006-2010,Álvaro Uribe Vélez,4221,414
2010-2014,Juan Manuel Santos,4848,528
2014-2018,Juan Manuel Santos,5575,676
2018-2022,Iván Duque Márquez,9965,1033
2022-2026,Gustavo Petro,3424,779


,tema,parrafos,porcentaje
0,Otros / sin tema claro,12589,44.9
1,Infraestructura y transporte,1521,5.4
2,Medio ambiente y cambio climático,1357,4.8
3,Educación,1247,4.4
4,"Ciencia, tecnología e innovación",1233,4.4
5,"Competitividad, industria, comercio y turismo",1162,4.1
6,"Equidad, género y grupos étnicos",1033,3.7
7,Seguridad y justicia,1012,3.6


,periodo,documento,pagina,texto
25398,2022-2026,plan-nacional-de-desarrollo-2022-2026-colombia...,289,Lo anterior integrará el accionar de la Mesa d...
15049,2018-2022,tomo-1-dnp,109,Se plantean las siguientes estrategias como re...
16611,2018-2022,tomo-1-dnp,386,Por su parte el MPC busca articular las políti...


## 4 · Índice FAISS (clase S14_1)
Embeddings con `HuggingFaceEmbeddings` (modelo multilingüe `multilingual-e5-small`) e índice `FAISS` de LangChain. Se calcula un embedding por párrafo:
la primera vez tarda varios minutos y después se reutiliza el índice guardado.

In [5]:
embeddings = indice.crear_embeddings()
vs = indice.cargar_o_construir(df, embeddings)
print("Párrafos indexados:", vs.index.ntotal)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (7 outputs left out)

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Construyendo el índice (la primera vez tarda varios minutos)...


Batches:   0%|          | 0/877 [00:00<?, ?it/s]

Párrafos indexados: 28033


## 5 · Buscar
Dos recuperadores sobre los mismos párrafos: **TF-IDF** (clase S13_2, busca palabras en común) y **denso** (clase S14_1, busca significado).
Si la pregunta nombra un gobierno (`2018-2022`, «Duque»), se busca solo en ese gobierno.

In [6]:
tfidf = recuperacion.crear_tfidf(df)

def tabla(resultados):
    return pd.DataFrame([{"periodo": r["periodo"], "doc": r["documento"], "pág.": r["pagina"],
                          "similitud": round(r["similitud"], 3), "texto": r["texto"][:200] + "…"} for r in resultados])

PREGUNTA = "¿Qué dice el plan 2018-2022 sobre el catastro multipropósito?"
periodos = recuperacion.detectar_periodos(PREGUNTA)
print("Gobierno detectado:", periodos)
print("\nDenso"); display(tabla(recuperacion.buscar_denso(vs, PREGUNTA, 3, periodos)))
print("TF-IDF"); display(tabla(recuperacion.buscar_tfidf(tfidf, PREGUNTA, 3, periodos)))

Gobierno detectado: ['2018-2022']

Denso


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,periodo,doc,pág.,similitud,texto
0,2018-2022,tomo-1-dnp,544,0.864,• El Instituto Geográfico Agustín Codazzi (IGA...
1,2018-2022,tomo-2-pnd,376,0.864,1282 Plan Nacional de Desarrollo 2018 - 2022 P...
2,2018-2022,tomo-2-pnd,320,0.863,1226 Plan Nacional de Desarrollo 2018 - 2022 P...


TF-IDF


,periodo,doc,pág.,similitud,texto
0,2018-2022,tomo-2-pnd,1036,0.200,3. La concertación en el marco del PND 2018 - ...
1,2018-2022,tomo-2-pnd,354,0.192,"• MinVivienda y el IGAC, en concurso con las e..."
2,2018-2022,tomo-1-dnp,871,0.165,A “Políticas e Armonizar la planeación para Mu...


## 6 · Evaluación independiente con el gold

### 6.1 El gold y su partición
`data/gold/preguntas_rag.csv` tiene preguntas `dominio` (con el `documento` y las `paginas` donde está la respuesta, buscadas **por una persona en el PDF**, no a partir de
lo que devuelve el sistema) y `fuera` (el sistema debe decir que no hay evidencia). La partición `dev`/`test` está congelada en `data/gold/split.csv`.

**Reglas:** con `dev` se toman *todas* las decisiones (umbral, k, qué recuperador mirar). `test` se mira **una vez**, al final. Si después de ver `test` se cambia algo, hay que decirlo.
Guía para completar el gold: `data/gold/README.md`.

In [23]:
import os

# Ruta al archivo con el problema del BOM
csv_path = "/content/An-lisis-documentos-DNP/dnp-rag/data/gold/preguntas_rag.csv"

# Corregir el BOM (Byte Order Mark) si está presente
if os.path.exists(csv_path):
    with open(csv_path, 'r', encoding='utf-8-sig') as f:
        contenido = f.read()
    with open(csv_path, 'w', encoding='utf-8') as f:
        f.write(contenido)
    print("✅ Se ha removido el BOM del archivo CSV para evitar el error de cabecera.")

gold = evaluacion.crear_split(evaluacion.cargar_gold())      # falla con un mensaje claro si el CSV está mal formado
display(evaluacion.resumen_gold(gold))

dev, test = gold[gold["split"] == "dev"], gold[gold["split"] == "test"]
MINIMO = 5     # preguntas de dominio completas que se piden en cada partición para calcular algo
LISTO_DEV = int(dev["completa"].sum()) >= MINIMO
LISTO_TEST = int(test["completa"].sum()) >= MINIMO
if not (LISTO_DEV and LISTO_TEST):
    print(f"\nAún faltan documento y páginas (hay {int(dev['completa'].sum())} listas en dev y {int(test['completa'].sum())} en test). "
          "Completa data/gold/preguntas_rag.csv y vuelve a ejecutar desde aquí.")

✅ Se ha removido el BOM del archivo CSV para evitar el error de cabecera.


preguntas  listas_para_evaluar
split tipo                                   
dev   dominio         26                 12.0
      fuera           45                  NaN
test  dominio         24                 18.0
      fuera            5                  NaN

### 6.2 DEV: ¿el denso supera a la línea base TF-IDF?
Métricas: **hit@k** = ¿apareció un párrafo de la página correcta entre los k primeros?; **MRR** = media de 1/posición del primero correcto.
Con tan pocas preguntas el intervalo de confianza es muy ancho (`IC95`): una diferencia pequeña **no** demuestra nada.

In [24]:
buscar_denso = lambda q, k, p: recuperacion.buscar_denso(vs, q, k, p)
buscar_tfidf = lambda q, k, p: recuperacion.buscar_tfidf(tfidf, q, k, p)

if LISTO_DEV:
    filas = {}
    for nombre, buscar in {"TF-IDF": buscar_tfidf, "Denso": buscar_denso}.items():
        for con_gobierno in (False, True):
            filas[nombre + (" + gobierno" if con_gobierno else "")] = evaluacion.evaluar_recuperacion(buscar, dev, usar_periodo=con_gobierno)
    display(pd.DataFrame(filas).T)
    print("Preguntas que fallan con el denso + gobierno (solo dev):")
    d = evaluacion.detalle_recuperacion(buscar_denso, dev)
    display(d[d["rango_correcto"].isna()])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (41 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,pregunta,esperado,rango_correcto,top1,similitud_top1
0,¿Cuántas instituciones de educación superior o...,"PND 2006–2010, tomo I p.288",None,PND_Tomo_1 p.288,0.906
1,¿En qué modalidad educativa se propuso inverti...,"PND 2006–2010, tomo II p.690",None,PND_Tomo_2 p.48,0.847
2,¿Qué dos componentes cubría la cuota mixta de ...,"PND 2006–2010, tomo I p.186",None,PND_Tomo_1 p.186,0.858
3,¿Qué cobertura de vacunación para todas las va...,"PND 2006–2010, tomo I p.134",None,PND_Tomo_1 p.134,0.862
4,¿Qué hospital nombró el PND 2006–2010 en la ac...,"PND 2006–2010, tomo II p.690",None,PND_Tomo_1 p.297,0.846
5,¿A quién ordenó girar directamente el artículo...,"PND 2010–2014, tomo II p.968",None,PND2010-2014 Tomo II CD p.428,0.888
6,¿Cuál era la meta 2014 de bibliotecarios capac...,"PND 2010–2014, tomo I p.408",None,PND2010-2014 Tomo I CD p.408,0.860
7,¿Qué dos instrumentos menciona la Ley 1450 de ...,"PND 2010–2014, tomo II p.955",None,PND2010-2014 Tomo II CD p.415,0.866
8,¿Qué meta anual de cobertura de vacunación en ...,"PND 2010–2014, tomo I p.388",None,PND2010-2014 Tomo I CD p.347,0.862
9,¿Qué meta fijó el PND 2010–2014 para el porcen...,"PND 2010–2014, tomo I p.388",None,PND2010-2014 Tomo II CD p.299,0.887


### 6.3 DEV: umbral de «no hay evidencia»
Si el mejor párrafo se parece poco a la pregunta, el sistema responde que no tiene evidencia. El umbral se calibra con las preguntas de `dev` (dominio vs. fuera de dominio) y se guarda.
No necesita páginas, solo las preguntas.

In [25]:
dev_dominio = dev[dev["tipo"] == "dominio"]["pregunta"]
dev_fuera = dev[dev["tipo"] == "fuera"]["pregunta"]
sim_dom_dev = evaluacion.similitudes_maximas(buscar_denso, dev_dominio)
sim_fue_dev = evaluacion.similitudes_maximas(buscar_denso, dev_fuera)

umbral, balanceada = evaluacion.calibrar_umbral(sim_dom_dev, sim_fue_dev)
rag.guardar_umbral(umbral)
print(f"Umbral calibrado en dev: {umbral:.3f} (exactitud balanceada {balanceada:.2f}; con tan pocas preguntas es una estimación optimista)")
display(pd.DataFrame({"dominio": pd.Series(sim_dom_dev).describe(), "fuera": pd.Series(sim_fue_dev).describe()}).round(3))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (63 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Umbral calibrado en dev: 0.856 (exactitud balanceada 0.85; con tan pocas preguntas es una estimación optimista)


,dominio,fuera
count,26.000,45.000
mean,0.873,0.836
std,0.015,0.024
min,0.846,0.787
25%,0.862,0.818
50%,0.875,0.832
75%,0.883,0.852
max,0.906,0.881


### 6.4 TEST: el resultado final (ejecutar **una sola vez**)
⚠️ No ejecutes esta celda hasta haber terminado de decidir todo con `dev`. Aquí no se mira el detalle de los fallos para no ajustar el sistema a `test`.

In [26]:
if LISTO_TEST:
    filas = {}
    for nombre, buscar in {"TF-IDF": buscar_tfidf, "Denso": buscar_denso}.items():
        for con_gobierno in (False, True):
            filas[nombre + (" + gobierno" if con_gobierno else "")] = evaluacion.evaluar_recuperacion(buscar, test, usar_periodo=con_gobierno)
    tabla_test = pd.DataFrame(filas).T
    display(tabla_test)
    sim_dom = evaluacion.similitudes_maximas(buscar_denso, test[test["tipo"] == "dominio"]["pregunta"])
    sim_fue = evaluacion.similitudes_maximas(buscar_denso, test[test["tipo"] == "fuera"]["pregunta"])
    print(evaluacion.evaluar_rechazo(sim_dom, sim_fue, rag.cargar_umbral()))
    tabla_test.to_csv(config.RESULTADOS_DIR / "resultados_test.csv")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (57 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'umbral': 0.856, 'rechaza_fuera_%': 60.0, 'rechaza_dominio_por_error_%': 4.2, 'n_dominio': 24, 'n_fuera': 5}


## 7 · El RAG completo
`rag.responder` hace: recuperar → ¿supera el umbral? → responder. Por defecto la respuesta es **extractiva**: frases literales de los PND, cada una con su cita `[n]` (clase S13_2),
así que no puede inventar. Opcionalmente un LLM redacta la respuesta con la cadena de LangChain de la clase S14_1 (`retriever | formatear → prompt → llm`).

In [27]:
for q in ["¿Qué metas de vivienda de interés social plantea el PND 2022-2026?",
          "¿Cómo se plantea reducir la informalidad laboral en el plan de Santos 2014-2018?",
          "vivienda",
          "¿Cuál es la receta tradicional de la bandeja paisa?"]:
    print("\n❓", q)
    print(rag.texto_chat(rag.responder(q, vs, embeddings)))


❓ ¿Qué metas de vivienda de interés social plantea el PND 2022-2026?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

• El Plan Nacional de Desarrollo 2022-2026 traza el camino para lograr el buen vivir de tal forma que toda la población logre coexistir sin miedo, con dignidad y con la garantía La Seguridad de sus derechos. [1]
• El Plan Nacional de Desarrollo 2022-2026: Colombia, Potencia Mundial de la Vida, que se expide por medio de la presente ley, tiene como objetivo sentar las bases para que el país se convierta en un líder de la protección de la vida a partir de la construcción de un nuevo contrato social que propicie la superación de injusticias y exclusiones históricas, la no repetición del conflicto, el cambio de nuestro relacionamiento con el ambiente y una transformación productiva sustentada en el conocimiento y en armonía con la naturaleza. [2]
• INTRODUCCIÓN l propósito del Plan Nacional de Desarrollo 2022- 2026 (PND 2022-2026), Colombia, Potencia Mundial de la Vida, es sentar las bases para que el país seEconvierta en un líder de la protección de la vida, a partir de la construcción de

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

• Se espera adicionalmente que el efecto del plan de inversiones de este Plan de Desarrollo permita la creación de cerca de 2,0 millones de empleos, con lo cual la tasa de desempleo terminaría en el 2018 en un 8,0 % y la informalidad laboral en las 13 áreas en un 41,8 %.7 [1]
• La estrategia macroeconómica para el periodo 2005-2018 es la de continuar reduciendo la tasa de desempleo, lo que permitirá alcanzar niveles del 8,0 % y seguir con la reducción de la informalidad, gracias a los estímulos señalados en la Ley 1607 de 2012. [3]
• De esta manera, las estrategias del Plan, de cara a estos propósitos, están orientadas a impactar en la productividad regional, reducir los costos de transporte de los productos rurales, y promover la equidad poblacional y territorial, mejorando las condiciones de conectividad y acceso de las poblaciones más necesitadas a los bienes y servicios para su desarrollo y bienestar. [4]

Fuentes:
1. PND 2014-2018 (Juan Manuel Santos), PND 2014-2018 Tomo 2 interne

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

No encontré evidencia suficiente en los Planes Nacionales de Desarrollo para responder esa pregunta. Prueba a reformularla o a nombrar un tema o un gobierno concretos.


In [28]:
USAR_LLM = False        # True: carga TinyLlama (clase S14_1). Necesita GPU. Con 1.100 M de parámetros el español es limitado.
llm = rag.crear_llm() if USAR_LLM else None
if llm:
    print(rag.texto_chat(rag.responder("¿Qué propone el PND 2022-2026 sobre transición energética?", vs, embeddings, llm=llm)))

## 8 · Calidad de las respuestas (calificación manual)
Se genera una hoja con las respuestas a preguntas de `test`. **Dos integrantes califican por separado** de 0 a 2:
**fidelidad** (¿todo lo que afirma está en las fuentes?), **relevancia** (¿responde lo preguntado?) y **citas** (¿cada [n] apunta a la fuente correcta?).
La hoja no se sobrescribe si ya existe, para no perder calificaciones.

In [29]:
ruta_rubrica = config.GOLD_DIR / "rubrica_respuestas.csv"
if not ruta_rubrica.exists():
    preguntas_rubrica = test[test["tipo"] == "dominio"]["pregunta"].head(10)
    evaluacion.crear_hoja_rubrica([(q, rag.responder(q, vs, embeddings, llm=llm)) for q in preguntas_rubrica], ruta_rubrica)
    print("Hoja creada:", ruta_rubrica, "-> califiquen las columnas *_A y *_B (0, 1 o 2) y vuelvan a ejecutar esta celda.")
else:
    hoja = pd.read_csv(ruta_rubrica)
    display(evaluacion.resumir_rubrica(hoja))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (19 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Hoja creada: /content/An-lisis-documentos-DNP/dnp-rag/data/gold/rubrica_respuestas.csv -> califiquen las columnas *_A y *_B (0, 1 o 2) y vuelvan a ejecutar esta celda.


## 9 · Chat (clase S14_2)
Es solo la interfaz del mismo RAG. Cada pregunta se responde de forma independiente (no recuerda la conversación).

In [30]:
from dnp_rag import app
app.lanzar_chat(vs, embeddings, llm=llm, compartir=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b86ae9d25784a7aa1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio Blocks instance: 35 backend functions
--------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.dataset.Dataset object at 0x7b41a64efe00>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7b41a64ef770>
fn_index=1
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7b41a64ef770>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7b41a64ef770>
 |-<gradio.components.state.State object at 0x7b41d9b1e210>
fn_index=2
 inputs:
 |-<gradio.components.state.State object at 0x7b41d9b1e210>
 |-<gradio.components.chatbot.Chatbot object at 0x7b41d9b90980>
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x7b41d9b90980>
fn_index=3
 inputs:
 |-<gradio.components.state.State object at 0x7b41d9b1e210>
 |-<gradio.components.state.State object at 0x7b41a5a896d0>
 outputs:
 |-<gradio.components.state.State object at 0x7b41d9b1e0d0>
 |-<gradio.components.chatbot.Chatbot object at 0x7b41d9b90980>
fn_index=4
 inputs:
 |-<gradio.components.

## 11 · Conclusiones
**Criterios aplicados en la evaluación**

- **Fidelidad:** El sistema RAG extrae fragmentos literales de los documentos sin inventar información; en la pregunta 6, donde decide no responder, no genera alucinaciones.

- **Relevancia:** Alexandra otorga 2 cuando el dato central solicitado aparece claramente en la respuesta aunque haya viñetas de contexto adicional; Carlos penaliza con 1 (filas 1, 5 y 8) cuando se incluye texto irrelevante (ruido) de extracción. Ambos coinciden en 0 en la fila 6, ya que, es un  falso negativo, pues, afirma no encontrar evidencia pese a haber recuperado páginas del capítulo de educación del PND 2014-2018.

- **Citas:** Todas las referencias [n] incluidas apuntan al tomo y página exactos de la lista de fuentes (puntaje 2), salvo en la fila 6 (0) donde la abstención omite cualquier cita.